# Phase 2 Curated Dataset EDA — AutoLens AI

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="deep")

PROJECT_ROOT = Path("..").resolve()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "dataset"
DATASET_ROOT = PROJECT_ROOT / "datasets"
EDA_ROOT = ARTIFACT_ROOT / "eda"
MANIFEST_ROOT = ARTIFACT_ROOT / "manifests"
SPLIT_ROOT = ARTIFACT_ROOT / "splits"
EXPORT_ROOT = EDA_ROOT / "notebook_exports"
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = [
    "SUV",
    "VAN",
    "STATION WAGON",
    "MICRO",
    "OPEN WHEEL / F1",
    "SEDAN",
    "HATCHBACK",
    "PICK UP",
]

REQUIRED_FILES = {
    "merged_manifest": MANIFEST_ROOT / "merged_manifest.csv",
    "all_splits": SPLIT_ROOT / "all_splits.csv",
    "balance_audit": EDA_ROOT / "balance_audit.csv",
    "status_counts": EDA_ROOT / "status_counts.csv",
    "source_by_class": EDA_ROOT / "source_by_class.csv",
    "review_candidates": EDA_ROOT / "review_candidates.csv",
    "duplicate_hashes": EDA_ROOT / "duplicate_hashes.csv",
    "missing_or_corrupt": EDA_ROOT / "missing_or_corrupt.csv",
    "micro_model_candidates": EDA_ROOT / "micro_model_candidates.csv",
    "micro_model_candidate_counts": EDA_ROOT / "micro_model_candidate_counts.csv",
    "near_duplicate_candidates": EDA_ROOT / "near_duplicate_candidates.csv",
    "near_duplicate_group_counts": EDA_ROOT / "near_duplicate_group_counts.csv",
    "near_duplicate_label_source_counts": EDA_ROOT / "near_duplicate_label_source_counts.csv",
    "decision_note": ARTIFACT_ROOT / "dataset_decision_note.md",
}

missing_artifacts = {name: path for name, path in REQUIRED_FILES.items() if not path.exists()}
if missing_artifacts:
    raise FileNotFoundError(f"Missing Phase 2 artifacts: {missing_artifacts}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"Notebook exports: {EXPORT_ROOT}")

## 1. Load Phase 2 artifacts

`merged_manifest.csv` contains every scanned local image. `all_splits.csv` contains the curated training-ready subset capped by the Phase 2 split generation policy.

In [ ]:
manifest = pd.read_csv(REQUIRED_FILES["merged_manifest"])
splits = pd.read_csv(REQUIRED_FILES["all_splits"])
balance_audit = pd.read_csv(REQUIRED_FILES["balance_audit"])
status_counts = pd.read_csv(REQUIRED_FILES["status_counts"])
source_by_class = pd.read_csv(REQUIRED_FILES["source_by_class"])
review_candidates = pd.read_csv(REQUIRED_FILES["review_candidates"])
duplicate_hashes = pd.read_csv(REQUIRED_FILES["duplicate_hashes"])
missing_or_corrupt = pd.read_csv(REQUIRED_FILES["missing_or_corrupt"])
micro_model_candidates = pd.read_csv(REQUIRED_FILES["micro_model_candidates"])
micro_model_candidate_counts = pd.read_csv(REQUIRED_FILES["micro_model_candidate_counts"])
near_duplicate_candidates = pd.read_csv(REQUIRED_FILES["near_duplicate_candidates"])
near_duplicate_group_counts = pd.read_csv(REQUIRED_FILES["near_duplicate_group_counts"])
near_duplicate_label_source_counts = pd.read_csv(REQUIRED_FILES["near_duplicate_label_source_counts"])

for df in (manifest, splits):
    for col in ["width", "height", "aspect_ratio", "file_size_bytes"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

usable = manifest.loc[manifest["split_eligible"].eq("yes")].copy()
curated = splits.copy()

summary = pd.DataFrame(
    {
        "metric": [
            "all_scanned_rows",
            "usable_unique_accepted_rows",
            "curated_split_rows",
            "review_candidate_rows",
            "duplicate_rows",
            "missing_or_corrupt_rows",
            "micro_model_candidate_rows",
            "near_duplicate_candidate_rows",
        ],
        "count": [
            len(manifest),
            len(usable),
            len(curated),
            len(review_candidates),
            len(duplicate_hashes),
            len(missing_or_corrupt),
            len(micro_model_candidates),
            len(near_duplicate_candidates),
        ],
    }
)
summary

## Final class counts

This is the direct final count table for the current Phase 2 dataset version after excluding `stanford-car-body-type-data` and applying the approved MICRO model whitelist.


In [ ]:
final_class_counts = (
    curated["normalized_label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("class")
    .reset_index(name="final_count")
)
final_class_counts.loc[len(final_class_counts)] = ["TOTAL", int(final_class_counts["final_count"].sum())]
final_class_counts.to_csv(EXPORT_ROOT / "final_class_counts.csv", index=False)
final_class_counts


In [ ]:
plt.figure(figsize=(12, 4))
plot_counts = final_class_counts.loc[final_class_counts["class"] != "TOTAL"].copy()
sns.barplot(data=plot_counts, x="class", y="final_count")
plt.title("Final Phase 2 class counts")
plt.xlabel("Class")
plt.ylabel("Images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "final_class_counts.png", dpi=160)
plt.show()


## 2. Schema, dtypes, nulls, and missing referenced files

This checks null-like values in the generated manifest/split tables and verifies whether referenced image paths still exist locally.

In [ ]:
def null_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    report = (
        df.isna()
        .sum()
        .rename("null_count")
        .to_frame()
        .assign(
            dataset=name,
            total_rows=len(df),
            null_percent=lambda x: (x["null_count"] / x["total_rows"] * 100).round(2),
            dtype=[str(dtype) for dtype in df.dtypes],
        )
        .reset_index(names="column")
        .sort_values(["null_count", "column"], ascending=[False, True])
    )
    return report[["dataset", "column", "dtype", "null_count", "total_rows", "null_percent"]]

nulls = pd.concat(
    [
        null_report(manifest, "merged_manifest"),
        null_report(usable, "usable_manifest"),
        null_report(curated, "curated_splits"),
    ],
    ignore_index=True,
)
nulls.to_csv(EXPORT_ROOT / "null_report.csv", index=False)
nulls.head(40)

In [ ]:
def with_file_exists(df: pd.DataFrame) -> pd.DataFrame:
    checked = df.copy()
    checked["absolute_path"] = checked["relative_path"].map(lambda p: DATASET_ROOT / str(p))
    checked["file_exists_now"] = checked["absolute_path"].map(lambda p: Path(p).exists())
    return checked

manifest_file_check = with_file_exists(manifest)
usable_file_check = with_file_exists(usable)
curated_file_check = with_file_exists(curated)

missing_file_summary = pd.DataFrame(
    {
        "dataset": ["merged_manifest", "usable_manifest", "curated_splits"],
        "rows": [len(manifest_file_check), len(usable_file_check), len(curated_file_check)],
        "missing_referenced_files": [
            (~manifest_file_check["file_exists_now"]).sum(),
            (~usable_file_check["file_exists_now"]).sum(),
            (~curated_file_check["file_exists_now"]).sum(),
        ],
    }
)
missing_file_summary.to_csv(EXPORT_ROOT / "missing_referenced_files.csv", index=False)
missing_file_summary

## 3. Mapping status: accepted vs review vs excluded

This section explains why some local images are usable and why others are not. The goal is to avoid training on labels that can mean multiple body types.

In [ ]:
status_pivot = (
    manifest.groupby(["mapping_status", "review_or_exclusion_reason"], dropna=False, observed=True)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
status_pivot.to_csv(EXPORT_ROOT / "mapping_status_reason_counts.csv", index=False)
status_pivot.head(30)

In [ ]:
plt.figure(figsize=(8, 4))
status_totals = manifest["mapping_status"].value_counts().rename_axis("mapping_status").reset_index(name="count")
sns.barplot(data=status_totals, x="mapping_status", y="count")
plt.title("Manifest rows by mapping status")
plt.xlabel("Mapping status")
plt.ylabel("Image rows")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "mapping_status_counts.png", dpi=160)
plt.show()

## 4. Class distribution for usable and final curated data

`usable` means safe, unique, accepted, split-eligible rows. `curated` means the uncapped split output that Phase 3 should consume.

In [ ]:
usable_class_counts = (
    usable["normalized_label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("class")
    .reset_index(name="usable_count")
)
curated_class_counts = (
    curated["normalized_label"]
    .value_counts()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("class")
    .reset_index(name="curated_count")
)
class_counts = usable_class_counts.merge(curated_class_counts, on="class", how="outer").merge(
    balance_audit[["class", "target_clean_count", "gap_to_target"]], on="class", how="left"
)
class_counts.to_csv(EXPORT_ROOT / "usable_and_curated_class_counts.csv", index=False)
class_counts

In [ ]:
plot_df = class_counts.melt(
    id_vars="class",
    value_vars=["usable_count", "curated_count", "target_clean_count"],
    var_name="series",
    value_name="count",
)
plt.figure(figsize=(13, 5))
sns.barplot(data=plot_df, x="class", y="count", hue="series")
plt.title("Class distribution: usable vs curated vs target")
plt.xlabel("Class")
plt.ylabel("Images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "class_distribution_usable_curated_target.png", dpi=160)
plt.show()

## 5. Split distribution

This confirms the train/validation/internal-test split per usable assignment class.

In [ ]:
split_distribution = (
    curated.groupby(["split", "normalized_label"], observed=True)
    .size()
    .reset_index(name="count")
)
split_distribution["normalized_label"] = pd.Categorical(
    split_distribution["normalized_label"], categories=TARGET_CLASSES, ordered=True
)
split_distribution = split_distribution.sort_values(["normalized_label", "split"])
split_distribution.to_csv(EXPORT_ROOT / "split_distribution.csv", index=False)
split_distribution.head(30)

In [ ]:
plt.figure(figsize=(13, 5))
sns.barplot(data=split_distribution, x="normalized_label", y="count", hue="split")
plt.title("Curated split distribution by class")
plt.xlabel("Class")
plt.ylabel("Images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "split_distribution_by_class.png", dpi=160)
plt.show()

## 6. Source-by-class coverage and source bias

This helps identify classes that come from too few sources, which can cause source/domain bias in Phase 3 training.

In [ ]:
usable_source_by_class = (
    usable.groupby(["source_id", "normalized_label"], observed=True)
    .size()
    .reset_index(name="count")
)
source_class_matrix = (
    usable_source_by_class.pivot_table(
        index="source_id",
        columns="normalized_label",
        values="count",
        aggfunc="sum",
        fill_value=0,
        observed=True,
    )
    .reindex(columns=TARGET_CLASSES, fill_value=0)
)
source_class_matrix.to_csv(EXPORT_ROOT / "usable_source_by_class_matrix.csv")
source_class_matrix

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(source_class_matrix, annot=True, fmt=".0f", cmap="Blues")
plt.title("Usable image coverage: source by class")
plt.xlabel("Class")
plt.ylabel("Source dataset")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "source_by_class_heatmap.png", dpi=160)
plt.show()

## 7. Duplicates, review candidates, and ambiguous labels

These images are intentionally **not** part of the usable training set unless they pass later manual review.

In [ ]:
review_reason_counts = (
    review_candidates["review_or_exclusion_reason"]
    .value_counts(dropna=False)
    .rename_axis("review_reason")
    .reset_index(name="count")
)
duplicate_source_counts = (
    duplicate_hashes["source_id"]
    .value_counts(dropna=False)
    .rename_axis("source_id")
    .reset_index(name="duplicate_count")
)
missing_corrupt_reason_counts = (
    missing_or_corrupt["review_or_exclusion_reason"]
    .value_counts(dropna=False)
    .rename_axis("reason")
    .reset_index(name="count")
)

review_reason_counts.to_csv(EXPORT_ROOT / "review_reason_counts.csv", index=False)
duplicate_source_counts.to_csv(EXPORT_ROOT / "duplicate_source_counts.csv", index=False)
missing_corrupt_reason_counts.to_csv(EXPORT_ROOT / "missing_corrupt_reason_counts.csv", index=False)

review_reason_counts.head(20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=review_reason_counts.head(12), y="review_reason", x="count", ax=axes[0])
axes[0].set_title("Top review/ambiguous label reasons")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("Reason")

sns.barplot(data=duplicate_source_counts, y="source_id", x="duplicate_count", ax=axes[1])
axes[1].set_title("Duplicate hash rows by source")
axes[1].set_xlabel("Duplicate rows")
axes[1].set_ylabel("Source")

plt.tight_layout()
plt.savefig(EXPORT_ROOT / "review_and_duplicate_summary.png", dpi=160)
plt.show()

## 8. Image dimensions, aspect ratios, and file-size quality checks

This section highlights image-quality distributions that affect Phase 3 preprocessing choices.

In [ ]:
quality_cols = ["width", "height", "aspect_ratio", "file_size_bytes"]
quality_summary = (
    usable[quality_cols]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .T
    .round(3)
)
quality_summary.to_csv(EXPORT_ROOT / "usable_image_quality_describe.csv")
quality_summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.ravel(), quality_cols):
    values = usable[col].dropna()
    if col == "file_size_bytes":
        values = values / 1024
        xlabel = "file_size_kb"
    else:
        xlabel = col
    sns.histplot(values, bins=50, ax=ax)
    ax.set_title(f"Usable {xlabel} distribution")
    ax.set_xlabel(xlabel)
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "usable_image_quality_histograms.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 13), sharex=True)
for ax, col in zip(axes, ["width", "height", "aspect_ratio"]):
    sns.boxplot(data=usable, x="normalized_label", y=col, ax=ax)
    ax.set_title(f"{col} by class")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "image_quality_by_class_boxplots.png", dpi=160)
plt.show()

## 9. Outlier candidates for manual review

These are accepted/usable images with unusual dimensions, aspect ratios, or very small file sizes. They are not automatically removed here; they are surfaced for review.

In [ ]:
def iqr_outliers(df: pd.DataFrame, col: str) -> pd.Series:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[col].lt(lower) | df[col].gt(upper)

outlier_flags = usable[["source_id", "relative_path", "normalized_label", *quality_cols]].copy()
outlier_flags["width_outlier"] = iqr_outliers(outlier_flags, "width")
outlier_flags["height_outlier"] = iqr_outliers(outlier_flags, "height")
outlier_flags["aspect_ratio_outlier"] = iqr_outliers(outlier_flags, "aspect_ratio")
outlier_flags["tiny_file"] = outlier_flags["file_size_bytes"].lt(outlier_flags["file_size_bytes"].quantile(0.01))
outlier_flags["any_outlier"] = outlier_flags[["width_outlier", "height_outlier", "aspect_ratio_outlier", "tiny_file"]].any(axis=1)

outlier_candidates = outlier_flags.loc[outlier_flags["any_outlier"]].copy()
outlier_candidates.to_csv(EXPORT_ROOT / "usable_outlier_candidates.csv", index=False)

outlier_summary = (
    outlier_candidates.groupby("normalized_label", observed=True)
    .size()
    .reindex(TARGET_CLASSES, fill_value=0)
    .rename_axis("class")
    .reset_index(name="outlier_candidate_count")
)
outlier_summary

## 10. Model-name MICRO candidate audit

This extra audit searches model names in `raw_label` and `relative_path` for micro-style cars similar to the provided MICRO example. This audit defines the final MICRO policy: only user-approved model whitelist matches are converted to MICRO; generic City Car is not used.

The current local matches are FIAT 500 and Smart Fortwo. No generic City Car rows are included.


In [ ]:
micro_model_candidate_counts.sort_values(["micro_candidate_tier", "micro_candidate_unique_valid", "count"], ascending=[True, True, False])


In [ ]:
micro_unique_valid = micro_model_candidates.loc[
    micro_model_candidates["micro_candidate_unique_valid"].eq("yes")
].copy()

micro_source_summary = (
    micro_unique_valid.groupby(["micro_candidate_tier", "micro_model_name", "source_id", "mapping_status", "review_or_exclusion_reason"], dropna=False, observed=True)
    .size()
    .reset_index(name="count")
    .sort_values(["micro_candidate_tier", "micro_model_name", "count"], ascending=[True, True, False])
)
micro_source_summary.to_csv(EXPORT_ROOT / "micro_model_candidate_source_summary.csv", index=False)
micro_source_summary


In [ ]:
plt.figure(figsize=(10, 4))
plot_micro = micro_model_candidate_counts.loc[
    micro_model_candidate_counts["micro_candidate_unique_valid"].eq("yes")
].copy()
sns.barplot(data=plot_micro, x="micro_model_name", y="count", hue="micro_candidate_tier")
plt.title("Unique-valid MICRO model-name candidates")
plt.xlabel("Model candidate")
plt.ylabel("Images")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "micro_model_candidate_counts.png", dpi=160)
plt.show()


## 11. Perceptual near-duplicate audit

Exact SHA-256 duplicates are already removed. This section uses the generated 64-bit average perceptual hash (`ahash64`) to surface **cross-source near-duplicate candidates**. These rows are not automatically removed; they are evidence for the pre-outlier/pre-near-duplicate vs post-cleaning comparison gate.


In [ ]:
near_duplicate_group_counts.head(30)


In [ ]:
near_duplicate_label_source_counts.sort_values("count", ascending=False).head(40)


In [ ]:
plt.figure(figsize=(12, 5))
near_plot = near_duplicate_label_source_counts.copy()
near_plot["normalized_label"] = near_plot["normalized_label"].fillna("UNMAPPED").replace({"": "UNMAPPED"})
sns.barplot(data=near_plot.groupby("normalized_label", as_index=False)["count"].sum(), x="normalized_label", y="count")
plt.title("Perceptual near-duplicate candidate rows by label")
plt.xlabel("Class")
plt.ylabel("Candidate rows")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(EXPORT_ROOT / "near_duplicate_candidates_by_label.png", dpi=160)
plt.show()


## 12. Optional sample grid from curated data

Run this cell to visually inspect a small deterministic sample from each usable class. If a class has no accepted images, it will be skipped.


In [ ]:
SAMPLES_PER_CLASS = 4
sample_rows = (
    curated.sort_values(["normalized_label", "sha256", "relative_path"])
    .groupby("normalized_label", group_keys=False, observed=True)
    .head(SAMPLES_PER_CLASS)
    .copy()
)

n_rows = sample_rows["normalized_label"].nunique()
n_cols = SAMPLES_PER_CLASS
fig, axes = plt.subplots(max(n_rows, 1), n_cols, figsize=(3.2 * n_cols, 2.8 * max(n_rows, 1)))
if n_rows == 1:
    axes = [axes]

for row_idx, (label, group) in enumerate(sample_rows.groupby("normalized_label", observed=True)):
    group = group.reset_index(drop=True)
    for col_idx in range(n_cols):
        ax = axes[row_idx][col_idx] if n_rows > 1 else axes[col_idx]
        ax.axis("off")
        if col_idx >= len(group):
            continue
        image_path = DATASET_ROOT / group.loc[col_idx, "relative_path"]
        try:
            image = Image.open(image_path).convert("RGB")
            ax.imshow(image)
            ax.set_title(f"{label}\n{group.loc[col_idx, 'source_id']}", fontsize=9)
        except Exception as exc:  # display path-level issue without failing entire EDA
            ax.set_title(f"Could not load\n{image_path.name}\n{exc}", fontsize=8)

plt.tight_layout()
plt.savefig(EXPORT_ROOT / "curated_sample_grid.png", dpi=160)
plt.show()